# Novas Tecnologias de Banco de Dados
Profa. Dra. Sahudy Montenegro González

População Mundial - Grupo 5

Ariel Sadetsky - 793240
 
Vanderlei Guilherme Andrade de Assis - 802162

Fase Final

Nosso planejamento para execução do projeto segue as etapas descritas pelo fluxograma da imagem seguinte que inicia pelo site do Worldometers (alvo de nosso scraping) passando pela limpeza e transformação dos dados adquiridos nesta fonte até a preparação desses dados para sua exportação e construção de um projeto físico e gráficos objetivando a análise planejada e descrita em nossas consultas:

![Imagem local](./_Fluxograma2.png)

Foram feitas algumas correções em nosso diagrama de Data Warehouse, sendo elas:

- Eliminação da tentativa de outrigger: os atributos dela foram incorporados em outras tabelas, principalmente pela nova criada.
- Criação da tabela Pais_ponte: foi utilizado o recurso avançado de tabela-ponte para otimizar a listagem de países vizinhos para cada país. Com ela, não será necessário o uso de um vetor de países vizinhos para cada país registrado, pois para cada tabela de Pais_ponte, podem haver várias de Pais_vizinho de acordo com a quantidade de vizinhos.
- As tabelas “Continente” e “Regiao” foram eliminadas.
- A dimensão de Causalidade e as mini-dimensões (antes criadas a partir desta tabela para organizar a causa de eventos) foi eliminada pela impossibilidade de sua execução devido à falta de dados de fácil extração e adaptáveis às outras dimensões do data warehouse.  

Segue nosso novo diagrama corrigido conforme descrito.

![Imagem local](./esquema_dw.png)

Explicação adicional do que é uma tabela ponte:

![Imagem local](./ExemploPonte.png)

A imagem acima obtida do endereço https://cursosriser.com.br/2021/04/23/pontes-relacionais-no-power-bi/ 

Exemplifica esse recurso que é usado para se tratar relacionamentos de muitos para muitos. Com a ponte, essa relação é descrita por duas relações de 1 para muitos, o que ajuda na filtragem de um fluxo isolado de dados, no nosso caso, para a dimensão de país.

Instalação das bibliotecas a serem utilizadas para a ETL:

In [0]:
#!pip install beautifulsoup4
#!pip install pandas
#!pip install lxml


Configurações de exibição da biblioteca pandas:

In [0]:
import pandas as pd

# Aumentar o número máximo de colunas exibidas
#pd.set_option('display.max_columns', None)  # Exibir todas as colunas

# Aumentar o número máximo de linhas exibidas
#pd.set_option('display.max_rows', None)  # Exibir todas as linhas

# Aumentar o número máximo de caracteres exibidos por coluna
#pd.set_option('display.max_colwidth', None)  # Exibir todos os caracteres de cada coluna

Segue-se com a captura do link da Web page que será alvo de nosso Scraping de forma a obtermos a principal tabela do site Wordometers:

In [0]:
import requests
from bs4 import BeautifulSoup

# resgata a url do worldometers
url = "https://www.worldometers.info/world-population/"
response = requests.get(url)

# Analiza a página
soup = BeautifulSoup(response.content, 'html.parser')

# Extrai a tabela correta
table = soup.find('table', {'class': 'table'}) 

# Verifique se a tabela foi encontrada
if table:
    print("Tabela encontrada!")
else:
    print("Tabela não encontrada.")

Extrai-se os dados da tabela para a criação do data frame:
Aqui headers é uma lista com as colunas da tabela extraída.
A data, por sua vez, é uma lista de listas contendo, em suas sublistas, os valores das linhas da tabela extraída, ou seja, dados para cada país individualmente.

In [0]:
import pandas as pd

# Extrair todas as linhas da tabela
rows = table.find_all('tr')

# Extrair cabeçalhos
headers = [header.text.strip() for header in rows[0].find_all('th')]

# Extrair dados
data = []
for row in rows[1:]:
    cells = row.find_all('td')
    data.append([cell.text.strip() for cell in cells])

# Criar DataFrame
df = pd.DataFrame(data, columns=headers)

# Exibir DataFrame
print(df)

Cria-se um dataframe a partir dos cabeçálhos da tabela de populações por país.
Para cada linha da tabela, os dados são extraídos e organizados, incluindo os links de cada país para se obter informações sobre as populações de cada um.


In [0]:
# 3. Extrair a tabela correta
# A tabela de população por país tem o ID 'popbycountry'
table = soup.find('table', {'id': 'popbycountry'})

# Verifique se a tabela foi encontrada
if table:
    print("Tabela encontrada!")
else:
    print("Tabela não encontrada.")

# 4. Extrair os dados da tabela
# Extrair cabeçalhos
headers = []
for header in table.find_all('th'):
    headers.append(header.text.strip())

# Adicionar uma coluna para os links
headers.append("Link") 

# Extrair linhas de dados
data = []
for row in table.find_all('tr')[1:]:  # Ignora a primeira linha (cabeçalho)
    cells = row.find_all('td')
    
    # Extrair o link do país
    link = ""
    country_cell = cells[1].find('a')  # A segunda célula contém o link
    if country_cell:
        link = "https://www.worldometers.info" + country_cell['href']
    
    # Extrair os dados das células
    row_data = [cell.text.strip() for cell in cells]
    row_data.append(link)  # Adicionar o link à linha de dados
    
    data.append(row_data)

# 5. Criar o DataFrame com os nomes das colunas
df = pd.DataFrame(data, columns=headers)

# 6. Exibir o DataFrame
print(df)

Segue-se o Data Frame com os links para todas as páginas contendo as informações individuais de cada país do mundo.

In [0]:
print(df['Link'])

Seguem as informações individuais obtidas para cada país dadas pelos links acima:

In [0]:
print(df.columns)

Agora, itera-se cada link achado para cada país e se extrai a tabela correspondente a cada país, armazenando esses dados em um dicionário de data frames. 

In [0]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Verifique se o DataFrame tem as colunas corretas
if 'Country (or dependency)' not in df.columns or 'Link' not in df.columns:
    raise ValueError("O DataFrame deve conter as colunas 'Country (or dependency)' e 'Link'.")

# Dicionário para armazenar os DataFrames por país
country_dataframes = {}

# Iterar por cada link
for index, row in df.iterrows():
    country = row['Country (or dependency)']  # Nome do país
    link = row['Link']  # Link correspondente
    
    if link:  # Verifica se o link não está vazio
        print(f"Acessando: {link}")
        
        # 1. Fazer a requisição HTTP
        response = requests.get(link)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # 2. Extrair a tabela desejada
        table = soup.find('table', {'class': 'table'})  
        
        if table:
            # 3. Extrair os cabeçalhos da tabela
            headers = [header.text.strip() for header in table.find_all('th')]
            
            # 4. Extrair os dados da tabela
            data = []
            for row in table.find_all('tr')[1:]:  # Ignora a primeira linha (cabeçalho)
                cells = row.find_all('td')
                data.append([cell.text.strip() for cell in cells])
            
            # 5. Criar um DataFrame temporário com os dados da tabela
            temp_df = pd.DataFrame(data, columns=headers)
            
            # 6. Formatar as colunas do DataFrame
            # Remover espaços em branco e caracteres especiais dos nomes das colunas
            temp_df.columns = temp_df.columns.str.strip()  # Remove espaços em branco
            temp_df.columns = temp_df.columns.str.replace('\n', ' ')  
            temp_df.columns = temp_df.columns.str.replace(r'[^\w\s]', '', regex=True)  # Remove caracteres especiais
            
            # Concatenar os nomes das colunas usando "_"
            temp_df.columns = temp_df.columns.str.replace(' ', '_')  
            
            # 7. Armazenar o DataFrame no dicionário, usando o nome do país como chave
            country_dataframes[country] = temp_df
        else:
            print(f"Nenhuma tabela encontrada em: {link}")
    else:
        print("Link vazio, pulando...")

# Exibir os países e DataFrames armazenados
for country, df in country_dataframes.items():
    print(f"País: {country}")
    print(df)
    print("\n") 

Exemplo de como se acessar o DataFrame de um país específico:

In [0]:
# Exemplo: Acessar o DataFrame da Índia
india_df = country_dataframes['India']
print("DataFrame da Índia:")
print(india_df)

Para cada DataFrame é criado um arquivo csv para uma análise melhor de países específicos.

In [0]:
import os

# Definir o diretório onde os arquivos CSV serão salvos
output_dir = './Paises_csv/'  

# Verificar se o diretório existe; se não, criar o diretório
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Iterar sobre os países e DataFrames no dicionário
for country, df in country_dataframes.items():
    # Criar o nome do arquivo CSV
    file_name = f"{country.replace('country', '_')}.csv"  # Substitui espaços por "_" no nome do país
    file_path = os.path.join(output_dir, file_name)  # Cria o caminho completo do arquivo
    
    # Salvar o DataFrame em um arquivo CSV
    df.to_csv(file_path, index=False)
    print(f"Arquivo '{file_name}' salvo em '{output_dir}'.")

print("Todos os arquivos CSV foram salvos com sucesso!")

Aqui, há uma estruturação a partir da extração de dados sobre continentes, divididos em regiões que são, por sua vez, divididas em países. Essa organização é feita e entendida por uma lista de dicionários.

In [0]:
import requests
from bs4 import BeautifulSoup

# 1. Acessar a página
url = "https://www.worldometers.info/population/world/"
try:
    response = requests.get(url)
    response.raise_for_status()  # Verifica se a requisição foi bem-sucedida
except requests.exceptions.RequestException as e:
    print(f"Erro ao acessar a página: {e}")
    exit()

soup = BeautifulSoup(response.content, 'html.parser')

# 2. Extrair os dados dos continentes e regiões
# Encontrar todas as seções de continentes
continents_sections = soup.find_all('div', {'class': 'col-md-6 noli'})

# Verificar se as seções foram encontradas
if not continents_sections:
    raise ValueError("Seções de continentes não encontradas na página.")

# 3. Estruturar os dados
continents_data = []

# Iterar sobre todas as seções de continentes
for continents_section in continents_sections:
    # Iterar sobre os continentes dentro de cada seção
    for continent_block in continents_section.find_all('li'):
        # Extrair o nome do continente
        continent_header = continent_block.find('h4')
        if not continent_header:
            continue  
        
        continent = continent_header.text.strip()
        
        # Extrair as regiões
        regions = []
        for region_link in continent_block.find_all('a'):
            region_name = region_link.text.strip()
            region_url = "https://www.worldometers.info" + region_link['href']
            
            # Acessar a página da região para extrair os países
            try:
                region_response = requests.get(region_url)
                region_response.raise_for_status()  
            except requests.exceptions.RequestException as e:
                print(f"Erro ao acessar a página da região {region_name}: {e}")
                continue  
            
            region_soup = BeautifulSoup(region_response.content, 'html.parser')
            
            # Encontrar a lista de países na página da região
            countries_list = region_soup.find('div', {'class': 'col-md-12 noli'})
            if not countries_list:
                continue 
            
            countries = []
            for country_link in countries_list.find_all('a'):
                country_name = country_link.text.strip().replace(' ', '_')  
                countries.append(country_name)
            
            # Adicionar a região e seus países à lista de regiões
            regions.append({
                'Region': region_name.replace(' ', '_'),  
                'Countries': countries
            })
        
        # Adicionar o continente e suas regiões à lista de continentes
        continents_data.append({
            'Continent': continent.replace(' ', '_'), 
            'Regions': regions
        })

# 4. Exibir os dados
for continent in continents_data:
    print(f"Continente: {continent['Continent']}")
    for region in continent['Regions']:
        print(f"  Região: {region['Region']}")
        print(f"    Países: {', '.join(region['Countries'])}")
    print("\n")  

Agora cria-se um DataFrame com a estrutura hierárquica descrita acima. 

In [0]:
# Criar um DataFrame a partir dos dados
data = []
for continent in continents_data:
    for region in continent['Regions']:
        for country in region['Countries']:
            data.append({
                'Continent': continent['Continent'],
                'Region': region['Region'],
                'Country': country
            })

df = pd.DataFrame(data)

# Exibir o DataFrame
print(df)

Este DataFrame também é transformado em um arquivo csv para melhor análise.

In [0]:
# Salvar o DataFrame em um arquivo CSV
df.to_csv('countries_data.csv', index=False)

print("Arquivo 'countries_data.csv' salvo com sucesso!")

Formulação do DataWarehouse pela construção das tabelas com SQL:

Segue a construção da dimensão de tempo:

In [0]:
%sql

DROP TABLE IF EXISTS Dim_Tempo;

CREATE TABLE Dim_Tempo (
    tempo_id BIGINT PRIMARY KEY,
    ano INTEGER,
    decada INTEGER,
    seculo INTEGER
) USING DELTA;

In [0]:
%sql
use catalog `workspace`; select * from `default`.`Pais_vizinho` limit 100;

Segue a construção da dimensão de país, descrito pela ponte feita em nosso diagrama:

In [0]:
%sql
DROP TABLE IF EXISTS Dim_Pais;

CREATE TABLE Dim_Pais (
    Nome_pais VARCHAR(255) PRIMARY KEY, 
    densidade INTEGER,
    pop_urbana INTEGER,
    cresc_anual FLOAT,
    rank_global INTEGER,
    taxa_pop_global FLOAT
) USING DELTA;

Segue a construção da tabela de Pais_vizinho que complementa a dimensão de Pais:

In [0]:
%sql
DROP TABLE IF EXISTS Pais_vizinho;

CREATE TABLE Pais_vizinho (
    Nome_pais VARCHAR(255), -- País principal
    Nome_pais_vizinho VARCHAR(255), -- País vizinho
    PRIMARY KEY (Nome_pais, Nome_pais_vizinho), -- Chave primária composta
    CONSTRAINT fk_pais FOREIGN KEY (Nome_pais) REFERENCES Dim_Pais(Nome_pais)
) USING DELTA;

Segue a construção de nossa tabela Fato (Fato_Populacao) com as métricas de pop_total, taxa_de_crescimento, global_rank, pop_urbana, taxa_pop_global, taxa_pop_urbana e taxa_pop_continente que possuem nomes autoexplicativos.

In [0]:
%sql
DROP TABLE IF EXISTS Fato_Populacao;

CREATE TABLE Fato_Populacao (
    tempo_id BIGINT, -- Referencia Dim_Tempo(tempo_id)
    Nome_pais VARCHAR(255), -- Referencia Dim_Pais(Nome_pais)
    Populacao_absoluta: INTEGER,
    taxa_de_crescimento FLOAT,
    global_rank INTEGER,
    pop_urbana INTEGER,
    taxa_pop_global FLOAT,
    taxa_pop_urbana FLOAT,
    CONSTRAINT fk_tempo_fato FOREIGN KEY (tempo_id) REFERENCES Dim_Tempo(tempo_id),
    CONSTRAINT fk_pais_fato FOREIGN KEY (Nome_pais) REFERENCES Dim_Pais(Nome_pais)
) USING DELTA;

Teste de funcionamento da criação das tabelas pela seleção de suas tuplas (ainda não foram populadas):

In [0]:
%sql
SELECT * FROM dim_tempo

In [0]:
%sql
SELECT * FROM dim_pais


In [0]:
%sql
SELECT * FROM pais_vizinho

In [0]:
%sql
SELECT * FROM Fato_Populacao


A partir de agora, todas as tabelas são populadas com os dados extraídos e transformados anteriormente.

No caso da dimensão tempo, os IDs são gerados automaticamente e há o cálculo de alguns atributos previstos em nosso modelo de DW como a década e o século. 

In [0]:
%sql
-- Inserção de Dados com Cálculo de Década e Século
INSERT INTO Dim_Tempo (tempo_id, ano, decada, seculo)
SELECT
    ROW_NUMBER() OVER (ORDER BY ano) AS tempo_id, -- Gera IDs automaticamente
    ano,
    FLOOR(ano / 10) * 10 AS decada,
    FLOOR((ano - 1) / 100) + 1 AS seculo 
FROM (
    SELECT explode(sequence(1955, 2024)) AS ano -- Gera anos de 1500 a 2023
);

In [0]:
%sql
SELECT * FROM dim_tempo

A seguir, lida-se com os dados da dimensão país inicialmente configurando uma tabela delta com spark. O código também lida com a padronização de informações, com possíveis falhas e com ausência de tuplas e colunas. Aqui, busca-se fazer a leitura dos arquivos CSVs para se extrair informações relevantes e já tratá-las e limpá-las para que, ao final, se tenha todos esses dados armazenados eficientemente em uma tabela Delta.



In [0]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, FloatType, StringType

# Configuração do Spark
spark = SparkSession.builder \
    .appName("Load CSV to Delta Table") \
    .getOrCreate()

# Função para converter strings com vírgulas, porcentagens ou 'N.A.' em float
def converter_para_float(valor):
    if isinstance(valor, str):
        valor = valor.replace('%', '').strip()
        valor = valor.replace(',', '.')
        
        # Verifica se a string contém 'n.a.' (case-insensitive)
        if 'n.a.' in valor.lower():
            return 0
        
        # Tenta converter para float
        try:
            return float(valor)
        except ValueError:
            return 0.0
    return float(valor)


pasta_csv = "/Workspace/Users/vanderleiassis@estudante.ufscar.br/Paises_csv"

arquivos_csv = [arquivo for arquivo in os.listdir(pasta_csv) if arquivo.endswith('.csv')]

# Itera sobre cada arquivo CSV
for arquivo in arquivos_csv:
    caminho_arquivo = os.path.join(pasta_csv, arquivo)
    
    # Lê o arquivo CSV
    df = pd.read_csv(caminho_arquivo)
    
    if 'Year' not in df.columns:
        print(f"Arquivo {arquivo} não contém a coluna 'Year'. Ignorando...")
        continue
    
    nome_pais = arquivo.replace('.csv', '')
    
    # Encontra a coluna que contém o rank global
    coluna_rank_global = None
    for coluna in df.columns:
        if "rank_global" in coluna.lower() or "global_rank" in coluna.lower():
            coluna_rank_global = coluna
            break
    
    if not coluna_rank_global:
        print(f"Arquivo {arquivo} não contém uma coluna de rank global. Ignorando...")
        continue
    
    # Prepara os dados para inserção
    dados_pais = {
        'Nome_pais': str(nome_pais),
        'densidade': int(converter_para_float(df['Density_PKm²'].iloc[0])),
        'pop_urbana': int(converter_para_float(df['Urban_Population'].iloc[0])),
        'cresc_anual': converter_para_float(df['Yearly___Change'].iloc[0]),
        'rank_global': int(converter_para_float(df[coluna_rank_global].iloc[0])),
        'taxa_pop_global': converter_para_float(df['Countrys_Share_of_World_Pop'].iloc[0]),
    }
    
    # Converte o dicionário para um DataFrame do Spark
    df_spark = spark.createDataFrame([dados_pais])
    
    # Define o esquema esperado para a tabela Delta
    schema = {
        'Nome_pais': StringType(),
        'densidade': IntegerType(),
        'pop_urbana': IntegerType(),
        'cresc_anual': FloatType(),
        'rank_global': IntegerType(),
        'taxa_pop_global': FloatType(),
    }
    
    # Converte as colunas para os tipos corretos
    for coluna, tipo in schema.items():
        df_spark = df_spark.withColumn(coluna, df_spark[coluna].cast(tipo))
    
    # Insere os dados na tabela Dim_Pais
    df_spark.write.format("delta").mode("append").saveAsTable("Dim_Pais")

print("Processo concluído.")

In [0]:
%sql
SELECT * FROM dim_pais 

O próximo trecho de código tem por objetivo criar e armazenar os pares de países vizinhos. Para isso ele segue algumas etapas, começando pelo carregamento do arquivo csv que contém os dados sobre países e suas regiões. Em seguida, os países são agrupados em suas respectivas regiões e são gerados os pares de países vizinhos dentro de cada região. Por fim, esses dados são armazenados na tabela Pais_vizinho. Aqui o Apache Spark é utilizado para garantir a escalabilidade e a distribuição dos dados.

In [0]:
import pandas as pd
from pyspark.sql import SparkSession

# Inicialize uma sessão do Spark
spark = SparkSession.builder.appName("CountriesData").getOrCreate()

csv_path = "/Workspace/Users/vanderleiassis@estudante.ufscar.br/countries_data.csv"
df = pd.read_csv(csv_path)

print(df.columns)

df = df[["Country", "Region"]]

# Agrupe os países por região
grouped = df.groupby("Region")

# Crie uma lista para armazenar os pares de países vizinhos
paises_vizinhos = []

# Itere sobre cada região
for region, group in grouped:
    paises = group["Country"].tolist()
    # Crie pares de países vizinhos
    for i in range(len(paises)):
        for j in range(i + 1, len(paises)):
            paises_vizinhos.append((paises[i], paises[j]))
            paises_vizinhos.append((paises[j], paises[i]))  # Relação bidirecional

# Converta a lista de pares em um DataFrame do Pandas
df_vizinhos = pd.DataFrame(paises_vizinhos, columns=["Nome_pais", "Nome_pais_vizinho"])

# Converta o DataFrame do Pandas para um DataFrame do Spark
spark_df = spark.createDataFrame(df_vizinhos)

# Crie a tabela Pais_vizinho no Databricks (se ainda não existir)
spark.sql("""
CREATE TABLE IF NOT EXISTS Pais_vizinho (
    Nome_pais VARCHAR(255),
    Nome_pais_vizinho VARCHAR(255),
    PRIMARY KEY (Nome_pais, Nome_pais_vizinho)
) USING DELTA
""")

# Insira os dados na tabela Pais_vizinho
spark_df.write.format("delta").mode("append").saveAsTable("Pais_vizinho")

print("Dados inseridos com sucesso na tabela Pais_vizinho!")

In [0]:
%sql
SELECT *
FROM pais_vizinho
ORDER BY Nome_pais ASC;

Ainda são feitos outros tratamentos para tuplas faltantes:

In [0]:
# Verifica o esquema da tabela Delta
spark.sql("DESCRIBE FORMATTED Dim_Pais").show(truncate=False)

O código a seguir deveria estar populando a tabela fato de população.

In [0]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, FloatType, StringType, StructType, StructField, LongType
from pyspark.sql.functions import col, udf

# Configuração do Spark
spark = SparkSession.builder \
    .appName("Load CSV to Fato_Populacao") \
    .getOrCreate()

def converter_para_float(valor):
    if isinstance(valor, str):
        valor = valor.replace('%', '').strip()
        valor = valor.replace(',', '.')
        if 'n.a.' in valor.lower():
            return None
        try:
            return float(valor)
        except ValueError:
            return 0.0
    return float(valor)

# Registra a função como UDF
converter_para_float_udf = udf(converter_para_float, FloatType())

# Define o esquema esperado para a tabela Fato_Populacao
schema = StructType([
    StructField("tempo_id", LongType(), False),
    StructField("Nome_pais", StringType(), False),
    StructField("pop_total", IntegerType(), False),
    StructField("taxa_de_crescimento", FloatType(), False),
    StructField("global_rank", IntegerType(), False),
    StructField("pop_urbana", IntegerType(), False),
    StructField("taxa_pop_global", FloatType(), False),
    StructField("taxa_pop_urbana", FloatType(), False),
])

pasta_csv = "/Workspace/Users/vanderleiassis@estudante.ufscar.br/Paises_csv"

arquivos_csv = [arquivo for arquivo in os.listdir(pasta_csv) if arquivo.endswith('.csv')]

for arquivo in arquivos_csv:
    caminho_arquivo = os.path.join(pasta_csv, arquivo)
    
    # Lê o arquivo CSV diretamente com PySpark
    df = spark.read.csv(caminho_arquivo, header=True, inferSchema=True)
    
    # Extrai o nome do país do nome do arquivo
    nome_pais = arquivo.replace('.csv', '')
    
    # Adiciona o nome do país ao DataFrame
    df = df.withColumn("Nome_pais", col("Nome_pais").cast(StringType()))
    
    # Converte as colunas necessárias usando a UDF
    df = df.withColumn("Population", converter_para_float_udf(col("Population")).cast(IntegerType())) \
           .withColumn("Yearly___Change", converter_para_float_udf(col("Yearly___Change")).cast(FloatType())) \
           .withColumn("Global_Rank", converter_para_float_udf(col("Global_Rank")).cast(IntegerType())) \
           .withColumn("Urban_Population", converter_para_float_udf(col("Urban_Population")).cast(IntegerType())) \
           .withColumn("Countrys_Share_of_World_Pop", converter_para_float_udf(col("Countrys_Share_of_World_Pop")).cast(FloatType())) \
           .withColumn("Urban_Pop_", converter_para_float_udf(col("Urban_Pop_")).cast(FloatType()))
    
    # Seleciona e renomeia as colunas para o esquema final
    df_final = df.select(
        col("Year").cast(LongType()).alias("tempo_id"),
        col("Nome_pais"),
        col("Population").alias("pop_total"),
        col("Yearly___Change").alias("taxa_de_crescimento"),
        col("Global_Rank").alias("global_rank"),
        col("Urban_Population").alias("pop_urbana"),
        col("Countrys_Share_of_World_Pop").alias("taxa_pop_global"),
        col("Urban_Pop_").alias("taxa_pop_urbana")
    )
    
    # Insere os dados na tabela Fato_Populacao
    df_final.write.format("delta").mode("append").saveAsTable("Fato_Populacao")

print("Processo concluído.")

- "Error: Public DBFS root is disabled. Access is denied on path: /Workspace/Users/vanderleiassis@estudante.ufscar.br/Paises_csv/India.csv"

o databricks removeu o nosso acesso a pasta onde estão os csv, o acesso que a gente tava usando até agora

In [0]:
%sql
--Consulta: Estime a população do Brasil de 2020, no ano pandêmico
SELECT popucao_absoluta
FROM Fato_Populacao NATURAL JOIN dim_pais NATURAL JOIN Dim_Tempo
WHERE Nome_pais = 'Brazil' 
AND ano = 2020

--Qual país tem a maior população da região Latin_America_AND_the_Caribbean? 
SELECT pais, populacao_absoluta
FROM Fato_Populacao NATURAL JOIN dim_pais NATURAL JOIN Pais_vizinho
WHERE Pais_vizinho.regiao = 'Latin_America_AND_the_Caribbean'
ORDER BY populacao_absoluta DESC
LIMIT 1;



Por fim, segue-se a relação das ferramentas, bibliotecas e frameworks usados ao longo do desenvolvimento do projeto. O fluxograma do projeto abaixo mostra todas as etapas desde a extração dos dados, tratamento e limpeza e construção do projeto físico como a etapa final deste processo.

![Imagem local](./_Fluxograma1-1.png)

Conclusão:

Apesar dos problemas com o databricks ao final do projeto que nos impediu de popular a tabela fato e executar as culsultas, a coleta dos dados e tratamento dessas informações já nos deu uma noção da dificuldade presente nessas tarefas. O processamento dos dados para futura análise tende infelizmente a sofrer muito do viés daqueles que coletam e tratam os dados e sentimos que devido á limitação que tivemos em lidar com muitas fontes, prejudicou e estreitou nossos resultados com relação ao tema. Em geral, pudemos conhecer o universo de BI e lidar com questões importantes desta área.